# 02.3 — From Notebook to Training Script

The FashionMNIST training from 02.2, turned into a script the way it's done at work:
a config file, validated settings, logging, a command line, checkpoints, early stopping, resuming and tests.
The code is in [`training-script/`](training-script); this notebook explains it and runs it.

## What's in this notebook

- 1\. Why move from a notebook to a script
- 2\. The project layout
- 3\. Config: YAML, Pydantic and overrides
- 4\. Logging instead of print
- 5\. The command line and the main guard
- 6\. Reproducibility
- 7\. The training loop: early stopping and checkpoints
- 8\. Running the script
- 9\. Resuming a run
- 10\. Testing ML code
- 11\. Trade-offs
- 12\. Interview questions
- 13\. In my own words

In [1]:
import inspect
import json
import subprocess
import sys
from pathlib import Path

from pydantic import ValidationError

SCRIPT_DIR = Path("training-script")
sys.path.insert(0, str(SCRIPT_DIR))
import train                                   # noqa: E402  (the script's functions, to show their code)


def run(*args: str) -> None:
    """Run a command inside training-script/, as you would in a terminal, and show its output."""
    result = subprocess.run([sys.executable, *args], cwd=SCRIPT_DIR, capture_output=True, text=True)
    print(result.stdout + result.stderr)
    result.check_returncode()

# 1. WHY MOVE FROM A NOTEBOOK TO A SCRIPT

Notebooks are great for exploring. Training that others rely on runs as a script.

| | Notebook | Script |
|---|---|---|
| Runs on a server or cloud GPU | awkward | `python train.py ...` |
| Runs on a schedule or in a pipeline | hard | easy (cron, Airflow, a job queue) |
| Hidden state | cells run out of order | runs top to bottom, every time |
| Code review and git diffs | noisy JSON | clean |
| Tests | hard | pytest |
| Changing settings | edit cells | a config file and command-line flags |

The usual flow: explore in a notebook, then move the code that works into modules and a script.

# 2. THE PROJECT LAYOUT

```text
training-script/
├── config.yaml        every setting of a run
├── data.py            download with checksums, the Dataset, the DataLoaders   (from 02.2)
├── model.py           the MLP                                                 (from 02.2)
├── train.py           config, logging, command line, training loop, checkpoints
├── test_training.py   fast tests (pytest)
└── checkpoints/       one folder per run: best.pt, last.pt, metrics.json, train.log, config.yaml   (not in git)
```

Each file has one job. `data.py` and `model.py` don't know about the command line, so they can be imported and tested on their own.
Bigger projects use the same idea with a `src/` package and a `pyproject.toml` (folder 14).

# 3. CONFIG: YAML, PYDANTIC AND OVERRIDES

- **All settings live in one YAML file**, not scattered through the code. Changing an experiment means changing the config, not the code.
- **Pydantic validates the config before training starts.** A typo or a negative learning rate fails in a second, not after an hour of training.
  `extra="forbid"` rejects unknown keys, which catches typos such as `learning_rate` instead of `lr`.
- **Command-line flags override the file** for quick experiments: `--epochs 3 --lr 0.003`.
- The final config is saved in each run's folder, so every result can be traced back to its exact settings.

In [2]:
print((SCRIPT_DIR / "config.yaml").read_text())
print(inspect.getsource(train.Config))
print(inspect.getsource(train.load_config))

# Settings for one training run. Paths are relative to this folder; run the script from here.
data_dir: ../../data/fashion-mnist
out_dir: checkpoints
seed: 0
device: auto          # auto, cpu, cuda or mps

epochs: 15
batch_size: 128
lr: 0.001
hidden: [256, 128]
dropout: 0.2

val_size: 6000
patience: 3           # early stopping: epochs without a better validation accuracy
num_workers: 2

class Config(BaseModel):
    """Every setting of a run. Unknown keys and invalid values fail before training starts."""

    model_config = ConfigDict(extra="forbid")

    data_dir: Path
    out_dir: Path = Path("checkpoints")
    seed: int = 0
    device: str = "auto"
    epochs: int = Field(15, ge=1)
    batch_size: int = Field(128, ge=1)
    lr: float = Field(1e-3, gt=0)
    hidden: list[int] = [256, 128]
    dropout: float = Field(0.2, ge=0, lt=1)
    val_size: int = Field(6000, gt=0)
    patience: int = Field(3, ge=1)
    num_workers: int = Field(2, ge=0)

def load_config(args: argparse.Namespace)

In [3]:
for bad in [{"data_dir": "data", "learning_rate": 0.01}, {"data_dir": "data", "lr": -1}, {"data_dir": "data", "dropout": 1.5}]:
    try:
        train.Config(**bad)
    except ValidationError as e:
        error = e.errors()[0]
        print(f"{str(bad):<45} -> {error['loc'][0]}: {error['msg']}")

{'data_dir': 'data', 'learning_rate': 0.01}   -> learning_rate: Extra inputs are not permitted
{'data_dir': 'data', 'lr': -1}                -> lr: Input should be greater than 0
{'data_dir': 'data', 'dropout': 1.5}          -> dropout: Input should be less than 1


All three mistakes are caught before any data is loaded: the unknown key, the negative learning rate and the out-of-range dropout.

# 4. LOGGING INSTEAD OF PRINT

`print` only writes to the screen, which is lost when a job runs on a server. The `logging` module adds timestamps and levels,
and can write to several places at once. The script logs to the console *and* to `train.log` in the run folder.

| Level | Use it for |
|---|---|
| `DEBUG` | detail you only want while debugging |
| `INFO` | normal progress: epoch results, checkpoints saved |
| `WARNING` | something odd that didn't stop the run |
| `ERROR` | something failed |

Experiment trackers such as MLflow (03.3) go one step further: metrics, configs and files from every run in one searchable place.

In [4]:
print(inspect.getsource(train.setup_logging))

def setup_logging(log_file: Path) -> None:
    """Log to the console and to a file in the run folder."""
    logger.handlers.clear()
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(message)s", datefmt="%H:%M:%S")
    for handler in (logging.StreamHandler(sys.stdout), logging.FileHandler(log_file)):
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    logger.setLevel(logging.INFO)



# 5. THE COMMAND LINE AND THE MAIN GUARD

`argparse` (in the standard library) turns flags into Python values and writes the `--help` text for you.
Libraries such as `typer` and `click` do the same with less code; Hydra combines config files and overrides for large projects.

**The main guard**, `if __name__ == "__main__": main()`, runs the training only when the file is executed, not when it is imported. It matters twice here:
- DataLoader workers (`num_workers > 0`) on macOS and Windows start fresh Python processes that **import** the script.
  Without the guard, each worker would start training again.
- Tests import functions from `train.py` without starting a run.

In [5]:
run("train.py", "--help")

usage: train.py [-h] --config CONFIG [--epochs EPOCHS] [--lr LR]
                [--batch-size BATCH_SIZE] [--run-name RUN_NAME]
                [--resume RESUME]

Train an MLP on FashionMNIST.

options:
  -h, --help            show this help message and exit
  --config CONFIG       YAML file with the run settings
  --epochs EPOCHS       override the number of epochs
  --lr LR               override the learning rate
  --batch-size BATCH_SIZE
                        override the batch size
  --run-name RUN_NAME   folder name for this run (default: date and time)
  --resume RESUME       continue from a last.pt checkpoint



# 6. REPRODUCIBILITY

Each run records what is needed to repeat it:

| What | How the script does it |
|---|---|
| random seeds | `set_seed` seeds Python, NumPy and PyTorch at the start |
| the batch order | the shuffle generator is reseeded with `seed + epoch` every epoch, so the order depends only on the epoch number (the same idea as `DistributedSampler.set_epoch`) |
| the same validation split | the split uses a fixed generator, independent of the run's seed |
| settings | the validated config is saved as `config.yaml` in the run folder |
| code version | the git commit is logged at the start |
| environment | the PyTorch version and the device are logged; `requirements.txt` pins the libraries |
| data version | the files are checked against fixed checksums |

Even with all of this, GPU runs can differ slightly between hardware and library versions. Exact repeats on a GPU also need
`torch.use_deterministic_algorithms(True)`, which is slower.

In [6]:
print(inspect.getsource(train.set_seed))
print(inspect.getsource(train.git_commit))

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

def git_commit() -> str:
    """The code version this run used, so a result can be traced back to its code."""
    try:
        result = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, check=True)
        return result.stdout.strip()
    except (OSError, subprocess.CalledProcessError):
        return "unknown"



# 7. THE TRAINING LOOP: EARLY STOPPING AND CHECKPOINTS

- **Early stopping:** stop when validation accuracy hasn't improved for `patience` epochs (3 here). It saves time and limits overfitting.
- **`best.pt`**, the model checkpoint, is written whenever validation improves. It holds the weights and config used for testing and serving.
- **`last.pt`**, the training checkpoint, is written every epoch: weights, optimizer, epoch, early-stopping counter, history,
  and the CPU and GPU random states (dropout on a GPU uses the GPU's own generator).
  If the job crashes or a cheap cloud "spot" GPU is taken away, training resumes from the last epoch instead of from zero.
- At the end the script reloads `best.pt`, scores the test set once and writes `metrics.json`.

In [7]:
print(inspect.getsource(train.main))

def main(argv: list[str] | None = None) -> None:
    args = parse_args(argv)
    cfg = load_config(args)
    run_dir = args.resume.parent if args.resume else cfg.out_dir / (args.run_name or datetime.now().strftime("%Y%m%d-%H%M%S"))
    run_dir.mkdir(parents=True, exist_ok=True)
    setup_logging(run_dir / "train.log")
    (run_dir / "config.yaml").write_text(yaml.safe_dump(cfg.model_dump(mode="json"), sort_keys=False))

    set_seed(cfg.seed)
    device = get_device(cfg.device)
    logger.info("run %s | torch %s | device %s | git %s", run_dir.name, torch.__version__, device, git_commit())

    shuffle_generator = torch.Generator()
    train_loader, val_loader, test_loader, mean, std = make_loaders(
        cfg.data_dir, cfg.batch_size, cfg.val_size, cfg.num_workers, shuffle_generator
    )
    model = MLP(cfg.hidden, cfg.dropout, mean, std).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.lr)
    loss_fn = nn.CrossEntropyLoss()
    start_epoch, best_acc, stale_epo

# 8. RUNNING THE SCRIPT

A full run with up to 40 epochs. Early stopping decides when to stop.

In [8]:
run("train.py", "--config", "config.yaml", "--epochs", "40", "--run-name", "full")

13:59:25 INFO run full | torch 2.15.0 | device mps | git f7863df
13:59:30 INFO epoch 1/40 train_loss 0.5394 val_loss 0.3921 val_acc 0.8565 (4.4s)
13:59:31 INFO epoch 2/40 train_loss 0.3939 val_loss 0.3481 val_acc 0.8670 (0.9s)
13:59:32 INFO epoch 3/40 train_loss 0.3550 val_loss 0.3290 val_acc 0.8803 (0.9s)
13:59:33 INFO epoch 4/40 train_loss 0.3331 val_loss 0.3246 val_acc 0.8812 (0.9s)
13:59:34 INFO epoch 5/40 train_loss 0.3154 val_loss 0.3139 val_acc 0.8867 (0.9s)
13:59:35 INFO epoch 6/40 train_loss 0.3024 val_loss 0.3099 val_acc 0.8880 (0.9s)
13:59:36 INFO epoch 7/40 train_loss 0.2906 val_loss 0.3221 val_acc 0.8840 (0.9s)
13:59:37 INFO epoch 8/40 train_loss 0.2776 val_loss 0.2993 val_acc 0.8887 (0.9s)
13:59:37 INFO epoch 9/40 train_loss 0.2697 val_loss 0.3129 val_acc 0.8863 (0.8s)
13:59:38 INFO epoch 10/40 train_loss 0.2609 val_loss 0.3017 val_acc 0.8912 (0.8s)
13:59:39 INFO epoch 11/40 train_loss 0.2534 val_loss 0.3162 val_acc 0.8903 (0.9s)
13:59:40 INFO epoch 12/40 train_loss 0.244

Validation accuracy peaks at 0.894 in epoch 14. Epochs 15–17 don't beat it, so early stopping ends the run at epoch 17 instead of 40, and the best weights score 0.889 on the test set.

The first epoch takes about 4 s and the rest under 1 s: starting the two DataLoader worker processes costs a few seconds once, and `persistent_workers=True` keeps them alive for the later epochs.

In [9]:
run_dir = SCRIPT_DIR / "checkpoints" / "full"
print("files in the run folder:", sorted(p.name for p in run_dir.iterdir()))
metrics = json.loads((run_dir / "metrics.json").read_text())
print(f"metrics.json: best epoch {metrics['best_epoch']}, val_acc {metrics['val_acc']:.4f}, test_acc {metrics['test_acc']:.4f}, {len(metrics['history'])} epochs of history")

files in the run folder: ['best.pt', 'config.yaml', 'last.pt', 'metrics.json', 'train.log']
metrics.json: best epoch 14, val_acc 0.8942, test_acc 0.8894, 17 epochs of history


# 9. RESUMING A RUN

A 2-epoch run that "stops", then continues to epoch 4 from `last.pt`. If resuming is exact, its four epochs must match
the first four epochs of the uninterrupted run in section 8 (same seed and settings).

In [10]:
run("train.py", "--config", "config.yaml", "--epochs", "2", "--run-name", "resume-demo")
run("train.py", "--config", "config.yaml", "--resume", "checkpoints/resume-demo/last.pt", "--epochs", "4")

uninterrupted = json.loads((SCRIPT_DIR / "checkpoints/full/metrics.json").read_text())["history"][:4]
resumed = json.loads((SCRIPT_DIR / "checkpoints/resume-demo/metrics.json").read_text())["history"]
for a, b in zip(uninterrupted, resumed):
    print(f"epoch {a['epoch']}: val_acc uninterrupted {a['val_acc']:.4f}, stopped and resumed {b['val_acc']:.4f} | identical: {a == b}")

13:59:50 INFO run resume-demo | torch 2.15.0 | device mps | git f7863df
13:59:56 INFO epoch 1/2 train_loss 0.5394 val_loss 0.3921 val_acc 0.8565 (4.7s)
13:59:57 INFO epoch 2/2 train_loss 0.3939 val_loss 0.3481 val_acc 0.8670 (0.9s)
13:59:58 INFO best epoch 2: val_acc 0.8670, test_acc 0.8608



14:00:02 INFO run resume-demo | torch 2.15.0 | device mps | git f7863df
14:00:03 INFO resumed from checkpoints/resume-demo/last.pt after epoch 2 (best val_acc 0.8670)
14:00:08 INFO epoch 3/4 train_loss 0.3550 val_loss 0.3290 val_acc 0.8803 (4.8s)
14:00:09 INFO epoch 4/4 train_loss 0.3331 val_loss 0.3246 val_acc 0.8812 (1.4s)
14:00:10 INFO best epoch 4: val_acc 0.8812, test_acc 0.8733

epoch 1: val_acc uninterrupted 0.8565, stopped and resumed 0.8565 | identical: True
epoch 2: val_acc uninterrupted 0.8670, stopped and resumed 0.8670 | identical: True
epoch 3: val_acc uninterrupted 0.8803, stopped and resumed 0.8803 | identical: True
epoch 4: val_acc uninterrupted 0.8812, stopped and resumed 0.8812 | identical: True


The resumed run picks up after epoch 2 with the same best score, and all four epochs are identical to the uninterrupted run. That needs three things: the optimizer state, the CPU and GPU random states, and a batch order that depends only on the epoch.

A common mistake is to save the DataLoader's generator state instead. With persistent workers, the DataLoader draws an extra seed whenever it starts new workers, so a resumed run gets a different batch order: in a test, the train loss at epoch 3 was 0.3546 instead of 0.3540. Reseeding with `seed + epoch` avoids the problem.

# 10. TESTING ML CODE

ML code can be wrong and still run: the loss goes down a bit, and nobody notices the bug. Fast tests catch most of it.

| Test | What it catches |
|---|---|
| output shape | wrong layer sizes, a forgotten `flatten` |
| dataset item types | float labels, unscaled pixels (02.1 section 15) |
| config validation | typos and impossible values |
| **overfit one batch** | the most useful single test: a working model must memorise 32 examples; if it can't, the loss, the optimizer or the data pipeline is broken |
| checkpoint round trip | weights or buffers that are not saved and restored |
| end-to-end smoke test | the whole script for one epoch on real data: it must finish and write its files |

In a real repository, GitHub Actions runs these tests on every pull request (folder 14).

In [11]:
import test_training                           # noqa: E402

print(inspect.getsource(test_training.test_model_can_overfit_one_batch))
run("-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider")

def test_model_can_overfit_one_batch():
    """If the model can't memorise 32 examples, something in the training code is broken."""
    torch.manual_seed(0)
    x, y = torch.rand(32, 28, 28), torch.randint(0, 10, (32,))
    model = MLP([64])
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    for _ in range(100):
        optimizer.zero_grad()
        loss = nn.functional.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
    assert loss.item() < 0.05



......                                                                   [100%]



All 6 tests pass in about 3 seconds, including the one-epoch smoke test on the real data.

# 11. TRADE-OFFS

| Decision | Options | What you gain and what you pay | Usual choice |
|---|---|---|---|
| Where training runs | notebook / script | notebook: fast to explore, but hidden state and hard to automate. Script: repeatable, testable, schedulable, but slower to explore in | explore in notebooks, train with scripts |
| Command line | `argparse` / `typer` or `click` / Hydra | argparse: no extra dependency, a little verbose. typer/click: less code. Hydra: config composition and sweeps, but more to learn | argparse or typer for small projects, Hydra for many experiments |
| Config | constants in code / YAML + Pydantic / Hydra + OmegaConf | constants: simple, but every change is a code change. YAML + Pydantic: one file, validated. Hydra: layered configs and overrides | YAML + Pydantic |
| Run records | `print` / `logging` / an experiment tracker | print: lost on servers. logging: timestamps, levels and files. Tracker: compare runs side by side, but needs a server or account | logging always, plus MLflow or W&B for experiments (03.3) |
| Checkpoint frequency | best only / every epoch / every N steps | best only: smallest, but can't resume. Every epoch: resumable. Every N steps: needed when one epoch takes hours | best + last every epoch; every N steps for long LLM runs |
| Early stopping patience | small / large | small: saves time, but may stop during a temporary plateau. Large: safer, more wasted epochs | 3–10 epochs, depending on how noisy validation is |
| Tests | unit tests only / plus an end-to-end smoke test | unit tests: fast. Smoke test: catches wiring bugs, but slower and needs data | both; the smoke test on a tiny slice of data in CI |

# 12. INTERVIEW QUESTIONS

### Q1. How do you turn notebook code into production training code, in 30 seconds?
Move the data, model and training code into modules; put every setting in a config file validated with Pydantic, with command-line overrides;
log to the console and a file; seed everything and save the config and git commit with each run; save a best checkpoint and a resumable last checkpoint;
add early stopping; and write tests, including an overfit-one-batch test, that run in CI.

### Q2. Why validate the config?
So mistakes fail immediately instead of after hours of training. Pydantic checks types and ranges, and `extra="forbid"` rejects unknown keys, which catches typos that would otherwise be silently ignored.

### Q3. Why `logging` instead of `print`?
Logging adds timestamps and levels, can write to a file and the console at once, and can be filtered or sent to a log system. Print output is lost when a job runs on a server.

### Q4. What does `if __name__ == "__main__"` do, and why does it matter for DataLoaders?
It runs the code only when the file is executed directly, not when it's imported. DataLoader workers on macOS and Windows import the main script;
without the guard, each worker would start the training again. Tests also need to import functions without starting a run.

### Q5. How do you make a training run reproducible and traceable?
Seed Python, NumPy and PyTorch (and the DataLoader's generator); fix the data split; save the exact config; log the git commit, library versions and device;
version or checksum the data. On GPUs, full determinism also needs deterministic algorithms, at some speed cost.

### Q6. What is early stopping, and which weights do you keep?
Stop training when the validation metric hasn't improved for a set number of epochs (the patience). Keep the weights from the best validation epoch, not the last one.

### Q7. What's the difference between `best.pt` and `last.pt`? Why save `last.pt` every epoch?
`best.pt` holds the best weights and config, for evaluation and serving. `last.pt` holds everything needed to resume: weights, optimizer, epoch, counters and the CPU and GPU random states.
Reseeding the shuffle with `seed + epoch` makes the batch order after a resume the same as without the stop.
Saving it every epoch means a crash or a reclaimed spot instance costs at most one epoch.

### Q8. How do you test ML code?
Shape and dtype tests, config validation, a checkpoint round trip, and an end-to-end smoke test on a little data. The most useful is overfitting one small batch:
a correct setup drives the loss close to zero, so if it can't, something is broken.

### Q9. Why is the first epoch slower when `num_workers > 0`?
Starting the worker processes takes time; on macOS each one starts a new Python process and imports PyTorch. `persistent_workers=True` keeps them alive between epochs, so only the first epoch pays.

### Q10. How do you run many experiments with different settings?
Override the config from the command line (or with Hydra), give each run its own folder, and log everything to an experiment tracker.
For hyperparameter searches, a tool such as Optuna or W&B Sweeps launches the runs.

### Q11. How would you run this training on a cloud GPU?
Package it in a Docker image with pinned dependencies, read the data from object storage, launch it as a job (a cloud ML service, Kubernetes, or a plain VM),
write checkpoints to durable storage so it can resume on spot instances, and track the run in MLflow or W&B.

### Q12. What would you log for every training run?
The config, git commit, library versions, device, data version, per-epoch train and validation metrics, the best epoch, the test result, and timing.

# 13. IN MY OWN WORDS

Write this from memory, without looking above:

- Why training code moves out of notebooks:
- What goes in the config, and why it is validated:
- The difference between best.pt and last.pt:
- What a run should record so it can be repeated:
- The overfit-one-batch test, and why it is useful: